# Data Analysis — 20 Preguntas de Negocio

Todas las consultas sobre `ANALYTICS.OBT_TRIPS` usando Spark.

## Imports y configuración

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE      = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_ANALYTICS    = os.environ.get("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")

spark = SparkSession.builder \
    .appName("P3_DataAnalysis") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .getOrCreate()

sf_options = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER, "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE, "sfSchema": SCHEMA_ANALYTICS,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE, "sfRole": SNOWFLAKE_ROLE,
    "use_arrow"  : "false",
    "sfCompress" : "on",
}

def query_obt(sql):
    """Helper para hacer queries a OBT_TRIPS sin cargar toda la tabla"""
    return spark.read \
        .format("net.snowflake.spark.snowflake") \
        .options(**sf_options) \
        .option("query", sql) \
        .load()

df = (
     spark.read.format("net.snowflake.spark.snowflake")
     .options(**sf_options)
     .option("dbtable", "OBT_TRIPS")
     .load()
)

df = df.toDF(*[c.lower() for c in df.columns])
print(df.columns)

['pickup_datetime', 'dropoff_datetime', 'pu_location_id', 'pu_zone', 'pu_borough', 'do_location_id', 'do_zone', 'do_borough', 'service_type', 'vendor_id', 'vendor_name', 'rate_code_id', 'rate_code_desc', 'payment_type', 'payment_type_desc', 'trip_type', 'passenger_count', 'trip_distance', 'store_and_fwd_flag', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'airport_fee', 'total_amount', 'run_id', 'ingested_at_utc', 'source_path', 'source_year', 'source_month', 'pickup_date', 'pickup_hour', 'dropoff_date', 'dropoff_hour', 'day_of_week', 'month', 'year', 'trip_duration_min', 'avg_speed_mph', 'tip_pct']


## a) Top 10 zonas de pickup por volumen mensual

In [ ]:
result = (df.groupby("pu_zone", "source_year", "source_month")
  .agg(F.count("*").alias("num_trips"))
  .orderBy(F.col("num_trips").desc())
  .limit(10)
)

result.show(truncate=False)

In [35]:
raw = query_obt("""
    SELECT pu_zone AS pickup_zone, source_year, source_month, COUNT(*) AS num_trips
    FROM OBT_TRIPS 
    GROUP BY pu_zone, source_year, source_month
""")

(raw.orderBy(F.col("num_trips").desc())
   .limit(10)
   .show(truncate=False))

+---------------------+-----------+------------+---------+
|PICKUP_ZONE          |SOURCE_YEAR|SOURCE_MONTH|NUM_TRIPS|
+---------------------+-----------+------------+---------+
|Upper East Side South|2015       |4           |497591   |
|Upper East Side South|2015       |5           |484530   |
|Upper East Side South|2015       |3           |473319   |
|Upper East Side South|2015       |10          |468351   |
|Upper East Side South|2015       |1           |467013   |
|Upper East Side South|2016       |5           |465129   |
|Midtown Center       |2015       |3           |464034   |
|Midtown Center       |2015       |4           |460812   |
|Upper East Side North|2015       |4           |457952   |
|Midtown Center       |2016       |3           |457732   |
+---------------------+-----------+------------+---------+



In [11]:
query_obt("""
    SELECT pu_zone AS pickup_zone, source_year, source_month, COUNT(*) AS num_trips
    FROM OBT_TRIPS 
    GROUP BY pu_zone, source_year, source_month
    ORDER BY num_trips DESC
    LIMIT 10
""").show(truncate=False)

+---------------------+-----------+------------+---------+
|PICKUP_ZONE          |SOURCE_YEAR|SOURCE_MONTH|NUM_TRIPS|
+---------------------+-----------+------------+---------+
|Upper East Side South|2015       |4           |497591   |
|Upper East Side South|2015       |5           |484530   |
|Upper East Side South|2015       |3           |473319   |
|Upper East Side South|2015       |10          |468351   |
|Upper East Side South|2015       |1           |467013   |
|Upper East Side South|2016       |5           |465129   |
|Midtown Center       |2015       |3           |464034   |
|Midtown Center       |2015       |4           |460812   |
|Upper East Side North|2015       |4           |457952   |
|Midtown Center       |2016       |3           |457732   |
+---------------------+-----------+------------+---------+



## b) Top 10 zonas de dropoff por volumen mensual

In [ ]:
result = (df.groupby("do_zone", "source_year", "source_month")
  .agg(F.count("*").alias("num_trips"))
  .orderBy(F.col("num_trips").desc())
  .limit(10)
)

result.show(truncate=False)

In [38]:
raw = query_obt("""
    SELECT do_zone AS pickup_zone, source_year, source_month, COUNT(*) AS num_trips
    FROM OBT_TRIPS 
    GROUP BY do_zone, source_year, source_month
""")

(raw.orderBy(F.col("num_trips").desc())
  .limit(10)
  .show(truncate=False))

+---------------------+-----------+------------+---------+
|PICKUP_ZONE          |SOURCE_YEAR|SOURCE_MONTH|NUM_TRIPS|
+---------------------+-----------+------------+---------+
|Midtown Center       |2015       |3           |506863   |
|Midtown Center       |2015       |4           |502317   |
|Midtown Center       |2015       |5           |480532   |
|Midtown Center       |2015       |6           |475067   |
|Midtown Center       |2015       |1           |472918   |
|Upper East Side North|2015       |4           |468557   |
|Midtown Center       |2016       |3           |467725   |
|Upper East Side North|2015       |5           |464671   |
|Midtown Center       |2015       |7           |463933   |
|Midtown Center       |2015       |2           |462124   |
+---------------------+-----------+------------+---------+



In [13]:
query_obt("""
    SELECT do_zone AS dropoff_zone, source_year, source_month, COUNT(*) AS num_trips
    FROM OBT_TRIPS 
    GROUP BY do_zone, source_year, source_month
    ORDER BY num_trips DESC
    LIMIT 10
""").show(truncate=False)

+---------------------+-----------+------------+---------+
|DROPOFF_ZONE         |SOURCE_YEAR|SOURCE_MONTH|NUM_TRIPS|
+---------------------+-----------+------------+---------+
|Midtown Center       |2015       |3           |506863   |
|Midtown Center       |2015       |4           |502317   |
|Midtown Center       |2015       |5           |480532   |
|Midtown Center       |2015       |6           |475067   |
|Midtown Center       |2015       |1           |472918   |
|Upper East Side North|2015       |4           |468557   |
|Midtown Center       |2016       |3           |467725   |
|Upper East Side North|2015       |5           |464671   |
|Midtown Center       |2015       |7           |463933   |
|Midtown Center       |2015       |2           |462124   |
+---------------------+-----------+------------+---------+



## c) Evolucion mensual de total_amount y tip_pct por borough

In [ ]:
result = (df.groupby("pu_borough", "source_year", "source_month")
  .agg(
       F.avg("total_amount").alias("avg_total_amount")
       F.avg("tip_pct").alias("tip_pct")
      )
  .orderBy("source_year", "source_month")
)  

result.show(truncate=False)

In [47]:
raw = query_obt("""
    SELECT pu_borough as pickup_borough, source_year, source_month, AVG(total_amount), AVG(tip_pct)
    FROM OBT_TRIPS
    GROUP BY pu_borough, source_year, source_month
""")

(raw.orderBy("source_year", "source_month")
    .show(truncate=False)
)  

+--------------+-----------+------------+-------------------+------------------+
|PICKUP_BOROUGH|SOURCE_YEAR|SOURCE_MONTH|"AVG(TOTAL_AMOUNT)"|"AVG(TIP_PCT)"    |
+--------------+-----------+------------+-------------------+------------------+
|EWR           |2015       |1           |78.90306944444444  |11.038812448293983|
|Manhattan     |2015       |1           |13.753997351929735 |9.264648811115931 |
|Unknown       |2015       |1           |15.929261109034679 |9.478541690163064 |
|Brooklyn      |2015       |1           |15.733848621729502 |8.924348542056784 |
|Queens        |2015       |1           |29.487667330380948 |7.3268675067081555|
|Bronx         |2015       |1           |12.92496090889298  |2.2147486837817736|
|N/A           |2015       |1           |55.33771479382016  |8.883220391087116 |
|Staten Island |2015       |1           |33.19493917274939  |6.037351125476132 |
|Brooklyn      |2015       |2           |16.233152122845034 |9.176376111395639 |
|Unknown       |2015       |

In [16]:
query_obt("""
    SELECT pu_borough as pickup_borough, source_year, source_month, AVG(total_amount), AVG(tip_pct)
    FROM OBT_TRIPS
    GROUP BY pu_borough, source_year, source_month
    ORDER BY source_year, source_month
""").show(truncate=False)

+--------------+-----------+------------+-------------------+------------------+
|PICKUP_BOROUGH|SOURCE_YEAR|SOURCE_MONTH|"AVG(TOTAL_AMOUNT)"|"AVG(TIP_PCT)"    |
+--------------+-----------+------------+-------------------+------------------+
|N/A           |2015       |1           |55.33771479382017  |8.883220391087116 |
|Brooklyn      |2015       |1           |15.733848621729502 |8.924348542056785 |
|Manhattan     |2015       |1           |13.753997351929733 |9.264648811115931 |
|Unknown       |2015       |1           |15.92926110903468  |9.478541690163064 |
|Bronx         |2015       |1           |12.92496090889298  |2.2147486837817736|
|Staten Island |2015       |1           |33.194939172749386 |6.037351125476132 |
|EWR           |2015       |1           |78.90306944444444  |11.038812448293983|
|Queens        |2015       |1           |29.487667330380948 |7.326867506708155 |
|Bronx         |2015       |2           |13.050129836114646 |1.818700086592947 |
|Manhattan     |2015       |

## d) Ticket promedio por service_type y mes

In [ ]:
result = (
   df.groupby("service_type", "source_year", "source_month")
     .agg(F.avg(total_amount).alias("avg_total_amount"))
     .orderBy("source_year", "source_month", "service_type")
)

result.show(truncate=False)

In [17]:
raw = query_obt("""
    SELECT service_type, source_year, source_month,
           AVG(total_amount) AS avg_total_amount
    FROM OBT_TRIPS
    GROUP BY service_type, source_year, source_month
""")

(raw.orderBy("source_year", "source_month", "service_type")
    .show(truncate=False)
)

+------------+-----------+------------+------------------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|AVG_TOTAL_AMOUNT  |
+------------+-----------+------------+------------------+
|green       |2015       |1           |14.817828474082965|
|yellow      |2015       |1           |15.116499666828378|
|green       |2015       |2           |14.52486821889793 |
|yellow      |2015       |2           |15.40880690303405 |
|green       |2015       |3           |14.611900800933888|
|yellow      |2015       |3           |15.858151838192766|
|green       |2015       |4           |14.853505500707596|
|yellow      |2015       |4           |16.044777028397068|
|green       |2015       |5           |15.289383590934083|
|yellow      |2015       |5           |16.435894402534835|
|green       |2015       |6           |14.998880924822995|
|yellow      |2015       |6           |16.387955289528406|
|green       |2015       |7           |14.952044618191364|
|yellow      |2015       |7           |16.15309072844553

In [19]:
query_obt("""
    SELECT service_type, source_year, source_month,
           AVG(total_amount) AS avg_total_amount
    FROM OBT_TRIPS
    GROUP BY service_type, source_year, source_month
    ORDER BY source_year, source_month, service_type
""").show(truncate=False)

+------------+-----------+------------+------------------+
|SERVICE_TYPE|SOURCE_YEAR|SOURCE_MONTH|AVG_TOTAL_AMOUNT  |
+------------+-----------+------------+------------------+
|green       |2015       |1           |14.817828474082965|
|yellow      |2015       |1           |15.116499666828378|
|green       |2015       |2           |14.52486821889793 |
|yellow      |2015       |2           |15.40880690303405 |
|green       |2015       |3           |14.611900800933892|
|yellow      |2015       |3           |15.858151838192766|
|green       |2015       |4           |14.853505500707596|
|yellow      |2015       |4           |16.04477702839707 |
|green       |2015       |5           |15.289383590934081|
|yellow      |2015       |5           |16.43589440253484 |
|green       |2015       |6           |14.998880924822995|
|yellow      |2015       |6           |16.38795528952841 |
|green       |2015       |7           |14.95204461819136 |
|yellow      |2015       |7           |16.15309072844553

## e) Viajes por hora del dia y dia de semana (picos)

In [ ]:
result = (
    df.groupby("day_of_week", "pickup_hour")
      .agg(F.count("*").alias("num_trips"))
      .orderBy("day_of_week", "pickup_hour")
)

result.show(truncate=False)

In [57]:
raw = query_obt("""
    SELECT day_of_week, pickup_hour, COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY day_of_week, pickup_hour
""")

(raw.orderBy("day_of_week", "pickup_hour")
      .show(truncate=False)
)

+-----------+-----------+---------+
|DAY_OF_WEEK|PICKUP_HOUR|NUM_TRIPS|
+-----------+-----------+---------+
|1          |0          |6672374  |
|1          |1          |5858031  |
|1          |2          |4563569  |
|1          |3          |3512951  |
|1          |4          |2280996  |
|1          |5          |1101288  |
|1          |6          |1220560  |
|1          |7          |1671976  |
|1          |8          |2479720  |
|1          |9          |3688495  |
|1          |10         |4973488  |
|1          |11         |5735948  |
|1          |12         |6286921  |
|1          |13         |6392929  |
|1          |14         |6494079  |
|1          |15         |6340485  |
|1          |16         |6239018  |
|1          |17         |6420497  |
|1          |18         |6497350  |
|1          |19         |5841382  |
+-----------+-----------+---------+
only showing top 20 rows



In [56]:
query_obt("""
    SELECT day_of_week, pickup_hour, COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY day_of_week, pickup_hour
    ORDER BY day_of_week, pickup_hour
""").show(truncate=False)

+-----------+-----------+---------+
|DAY_OF_WEEK|PICKUP_HOUR|NUM_TRIPS|
+-----------+-----------+---------+
|1          |0          |6672374  |
|1          |1          |5858031  |
|1          |2          |4563569  |
|1          |3          |3512951  |
|1          |4          |2280996  |
|1          |5          |1101288  |
|1          |6          |1220560  |
|1          |7          |1671976  |
|1          |8          |2479720  |
|1          |9          |3688495  |
|1          |10         |4973488  |
|1          |11         |5735948  |
|1          |12         |6286921  |
|1          |13         |6392929  |
|1          |14         |6494079  |
|1          |15         |6340485  |
|1          |16         |6239018  |
|1          |17         |6420497  |
|1          |18         |6497350  |
|1          |19         |5841382  |
+-----------+-----------+---------+
only showing top 20 rows



## f) p50/p90 de trip_duration_min por borough de pickup

In [ ]:
result = (
    df.groupby("pu_borough")
      .agg(
          F.round(F.percentile_approx("trip_duration_min", 0.5), 2).alias("p50_duration"),
          F.round(F.percentile_approx("trip_duration_min", 0.9), 2).alias("p90_duration")
      ) 
      .withColumn("ratio", F.round(F.col("p50_duration") / F.col("p90_duration"), 2))
      .orderBy(F.col("p50_duration").desc())
)

result.show(truncate=False)

In [3]:
raw = query_obt("""
    SELECT pu_borough,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p50_duration,
           ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p90_duration,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_duration_min) / PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY trip_duration_min), 2) as ratio
    FROM OBT_TRIPS
    GROUP BY pu_borough
""")

(raw.orderBy(F.col("p50_duration").desc())
   .show(truncate=False)
)

+-------------+------------+------------+-----+
|PU_BOROUGH   |P50_DURATION|P90_DURATION|RATIO|
+-------------+------------+------------+-----+
|Queens       |24.62       |54.67       |0.45 |
|Staten Island|23.0        |70.45       |0.33 |
|Bronx        |13.73       |39.27       |0.35 |
|Brooklyn     |13.13       |33.23       |0.4  |
|Manhattan    |10.87       |25.17       |0.43 |
|Unknown      |10.52       |28.1        |0.37 |
|N/A          |1.12        |59.98       |0.02 |
|EWR          |0.28        |1.7         |0.17 |
+-------------+------------+------------+-----+



In [2]:
query_obt("""
    SELECT pu_borough,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p50_duration,
           ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p90_duration,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY trip_duration_min) / PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY trip_duration_min), 2) as ratio
    FROM OBT_TRIPS
    GROUP BY pu_borough
    ORDER BY p50_duration DESC
""").show(truncate=False)

+-------------+------------+------------+-----+
|PU_BOROUGH   |P50_DURATION|P90_DURATION|RATIO|
+-------------+------------+------------+-----+
|Queens       |24.62       |54.67       |0.45 |
|Staten Island|23.0        |70.45       |0.33 |
|Bronx        |13.73       |39.27       |0.35 |
|Brooklyn     |13.13       |33.23       |0.4  |
|Manhattan    |10.87       |25.17       |0.43 |
|Unknown      |10.52       |28.1        |0.37 |
|N/A          |1.12        |59.98       |0.02 |
|EWR          |0.28        |1.7         |0.17 |
+-------------+------------+------------+-----+



## g) avg_speed_mph por franja horaria (6-9, 17-20) y borough

In [ ]:
result = (
    df.withColumn("pickup_period",
           F.when(F.col("pickup_hour").between(6, 9), "morning")
            .when(F.col("pickup_hour").between(17, 20), "evening")
      )
      .filter(F.col("pickup_period").isNotNull())
      .groupby("pu_borough", "pickup_period")
      .agg(F.round(F.avg("avg_speed_mph"), 2).alias("avg_speed"))
      .orderBy("pu_borough", "pickup_period")
)

result.show(truncate=False)

In [5]:
raw = query_obt("""
    SELECT pu_borough AS borough, AVG(avg_speed_mph),
        CASE 
            WHEN pickup_hour BETWEEN 6 AND 9 THEN 'morning'
            WHEN pickup_hour BETWEEN 17 AND 20 THEN 'evening' 
        END AS pickup_period
    FROM OBT_TRIPS
    WHERE (pickup_hour BETWEEN 6 AND 9) 
    OR (pickup_hour BETWEEN 17 AND 20)
    GROUP BY pu_borough, pickup_period
""")

(raw.orderBy("borough", "pickup_period")
   .show(truncate=False)
)

+-------------+--------------------+-------------+
|BOROUGH      |"AVG(AVG_SPEED_MPH)"|PICKUP_PERIOD|
+-------------+--------------------+-------------+
|Bronx        |50.374576050969935  |evening      |
|Bronx        |143.3035068017433   |morning      |
|Brooklyn     |25.09432536720082   |evening      |
|Brooklyn     |54.721002594448784  |morning      |
|EWR          |775.1212763166562   |evening      |
|EWR          |767.1463911375595   |morning      |
|Manhattan    |38.53592091623219   |evening      |
|Manhattan    |22.120038777323018  |morning      |
|N/A          |601.9704817786999   |evening      |
|N/A          |638.5597632611498   |morning      |
|Queens       |56.27125374285516   |evening      |
|Queens       |363.00201172398477  |morning      |
|Staten Island|128.50111158910855  |evening      |
|Staten Island|95.62684281167692   |morning      |
|Unknown      |19.451365340866875  |evening      |
|Unknown      |38.557824099470665  |morning      |
+-------------+----------------

In [6]:
query_obt("""
    SELECT pu_borough AS borough, AVG(avg_speed_mph),
        CASE 
            WHEN pickup_hour BETWEEN 6 AND 9 THEN 'morning'
            WHEN pickup_hour BETWEEN 17 AND 20 THEN 'evening' 
        END AS pickup_period
    FROM OBT_TRIPS
    WHERE (pickup_hour BETWEEN 6 AND 9) 
    OR (pickup_hour BETWEEN 17 AND 20)
    GROUP BY pu_borough, pickup_period
    ORDER BY pu_borough, pickup_period
""").show(truncate=False)

+-------------+--------------------+-------------+
|BOROUGH      |"AVG(AVG_SPEED_MPH)"|PICKUP_PERIOD|
+-------------+--------------------+-------------+
|Bronx        |50.37457605096993   |evening      |
|Bronx        |143.3035068017433   |morning      |
|Brooklyn     |25.09432536720082   |evening      |
|Brooklyn     |54.721002594448784  |morning      |
|EWR          |775.1212763166562   |evening      |
|EWR          |767.1463911375594   |morning      |
|Manhattan    |38.535920916232186  |evening      |
|Manhattan    |22.12003877732302   |morning      |
|N/A          |601.9704817786998   |evening      |
|N/A          |638.5597632611499   |morning      |
|Queens       |56.27125374285516   |evening      |
|Queens       |363.00201172398477  |morning      |
|Staten Island|128.50111158910855  |evening      |
|Staten Island|95.62684281167694   |morning      |
|Unknown      |19.45136534086687   |evening      |
|Unknown      |38.557824099470665  |morning      |
+-------------+----------------

## h) Participacion por payment_type_desc y su relacion con tip_pct

In [ ]:
result = (
    df.groupby("payment_type_desc")
      .agg(
          F.count("*").alias("num_trips"),
          F.round(F.avg("tip_pct"), 2)
      )
      .withColumn("pct_trips", F.round((F.count("*") * 100)/ F.sum(F.count(*).over()), 2))
      .orderBy(F.col("num_trips").desc())
)

result.show(truncate=False)

In [8]:
raw = query_obt("""
    SELECT payment_type_desc,
           COUNT(*) as num_trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct_trips,
           ROUND(AVG(tip_pct), 2) as avg_tip_pct
    FROM OBT_TRIPS
    WHERE payment_type_desc IS NOT NULL
    GROUP BY payment_type_desc
""")

(raw.orderBy(F.col("num_trips").desc())
   .show(truncate=False)
)

+-----------------+---------+---------+-----------+
|PAYMENT_TYPE_DESC|NUM_TRIPS|PCT_TRIPS|AVG_TIP_PCT|
+-----------------+---------+---------+-----------+
|Credit card      |581349909|67.23    |15.05      |
|Cash             |256613320|29.67    |0.0        |
|Flex Fare trip   |20915809 |2.42     |13.27      |
|No charge        |3566575  |0.41     |0.02       |
|Dispute          |2313854  |0.27     |0.03       |
|Unknown          |3048     |0.00     |0.1        |
+-----------------+---------+---------+-----------+



In [9]:
query_obt("""
    SELECT payment_type_desc,
           COUNT(*) as num_trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct_trips,
           ROUND(AVG(tip_pct), 2) as avg_tip_pct
    FROM OBT_TRIPS
    WHERE payment_type_desc IS NOT NULL
    GROUP BY payment_type_desc
    ORDER BY num_trips DESC
""").show(truncate=False)

+-----------------+---------+---------+-----------+
|PAYMENT_TYPE_DESC|NUM_TRIPS|PCT_TRIPS|AVG_TIP_PCT|
+-----------------+---------+---------+-----------+
|Credit card      |581349909|67.23    |15.05      |
|Cash             |256613320|29.67    |0.0        |
|Flex Fare trip   |20915809 |2.42     |13.27      |
|No charge        |3566575  |0.41     |0.02       |
|Dispute          |2313854  |0.27     |0.03       |
|Unknown          |3048     |0.00     |0.1        |
+-----------------+---------+---------+-----------+



## i) Rate codes con mayor trip_distance y total_amount

In [ ]:
result = (
    df.groupby("rate_code_desc")
      .agg(
          F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
          F.round(F.avg("total_amount"), 2).alias("avg_total"),
          F.round(F.avg("trip_distance"), 2).alias("sum_distance"),
          F.round(F.avg("total_amount"), 2).alias("sum_total")
          
      )
      .orderBy(F.col("sum_total").desc())
)

result.show(truncate=False)

In [11]:
raw = query_obt("""
    SELECT rate_code_desc,
           ROUND(SUM(trip_distance), 2) as sum_distance,
           ROUND(AVG(trip_distance), 2) as avg_distance,
           ROUND(SUM(total_amount), 2) as sum_total,
           ROUND(AVG(total_amount), 2) as avg_total,
    FROM OBT_TRIPS
    GROUP BY rate_code_desc
""")

(raw.orderBy(F.col("sum_total").desc())
   .show(truncate=False)
)

+---------------------+---------------+------------+-----------------+---------+
|RATE_CODE_DESC       |SUM_DISTANCE   |AVG_DISTANCE|SUM_TOTAL        |AVG_TOTAL|
+---------------------+---------------+------------+-----------------+---------+
|Standard rate        |3.56326648518E9|4.38        |1.363290740549E10|16.74    |
|JFK                  |4.3457397314E8 |21.82       |1.43125199504E9  |71.88    |
|NULL                 |8.3636832564E8 |36.64       |5.9787559377E8   |26.19    |
|Negotiated fare      |9.203992199E7  |17.24       |3.0998826589E8   |58.07    |
|Newark               |2.889969681E7  |16.36       |1.6647922735E8   |94.26    |
|Nassau or Westchester|2.124515857E7  |28.83       |7.548856091E7    |102.45   |
|Null/unknown         |3.29063104E7   |19.82       |6.306045796E7    |37.99    |
|Group ride           |8941.68        |1.25        |187292.53        |26.17    |
+---------------------+---------------+------------+-----------------+---------+



In [12]:
query_obt("""
    SELECT rate_code_desc,
           ROUND(SUM(trip_distance), 2) as sum_distance,
           ROUND(AVG(trip_distance), 2) as avg_distance,
           ROUND(SUM(total_amount), 2) as sum_total,
           ROUND(AVG(total_amount), 2) as avg_total,
    FROM OBT_TRIPS
    GROUP BY rate_code_desc
    ORDER BY sum_total DESC
""").show(truncate=False)

+---------------------+---------------+------------+-----------------+---------+
|RATE_CODE_DESC       |SUM_DISTANCE   |AVG_DISTANCE|SUM_TOTAL        |AVG_TOTAL|
+---------------------+---------------+------------+-----------------+---------+
|Standard rate        |3.56326648518E9|4.38        |1.363290740549E10|16.74    |
|JFK                  |4.3457397314E8 |21.82       |1.43125199504E9  |71.88    |
|NULL                 |8.3636832564E8 |36.64       |5.9787559377E8   |26.19    |
|Negotiated fare      |9.203992199E7  |17.24       |3.0998826589E8   |58.07    |
|Newark               |2.889969681E7  |16.36       |1.6647922735E8   |94.26    |
|Nassau or Westchester|2.124515857E7  |28.83       |7.548856091E7    |102.45   |
|Null/unknown         |3.29063104E7   |19.82       |6.306045796E7    |37.99    |
|Group ride           |8941.68        |1.25        |187292.53        |26.17    |
+---------------------+---------------+------------+-----------------+---------+



## j) Mix yellow vs green por mes y borough

In [ ]:
result = (
    df.groupby("pu_borough", "source_year", "source_month", "service_type")
      .agg(
          F.count("*").alias("num_trips")
      )
      .orderBy("sum_total", "source_month", "service_type")
)

result.show(truncate=False)

In [14]:
raw = query_obt("""
    SELECT pu_borough, source_year, source_month, service_type,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, source_year, source_month, service_type
""")

(raw.orderBy("source_year", "source_month", "service_type")
   .show(truncate=False)
)

+-------------+-----------+------------+------------+---------+
|PU_BOROUGH   |SOURCE_YEAR|SOURCE_MONTH|SERVICE_TYPE|NUM_TRIPS|
+-------------+-----------+------------+------------+---------+
|Bronx        |2015       |1           |green       |91410    |
|EWR          |2015       |1           |green       |33       |
|Brooklyn     |2015       |1           |green       |569352   |
|N/A          |2015       |1           |green       |844      |
|Manhattan    |2015       |1           |green       |429801   |
|Queens       |2015       |1           |green       |412150   |
|Unknown      |2015       |1           |green       |2456     |
|Staten Island|2015       |1           |green       |222      |
|Queens       |2015       |1           |yellow      |636494   |
|Brooklyn     |2015       |1           |yellow      |229838   |
|Bronx        |2015       |1           |yellow      |9636     |
|Manhattan    |2015       |1           |yellow      |11610665 |
|Unknown      |2015       |1           |

In [15]:
query_obt("""
    SELECT pu_borough, source_year, source_month, service_type,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, source_year, source_month, service_type
    ORDER BY source_year, source_month, service_type
""").show(truncate=False)

+-------------+-----------+------------+------------+---------+
|PU_BOROUGH   |SOURCE_YEAR|SOURCE_MONTH|SERVICE_TYPE|NUM_TRIPS|
+-------------+-----------+------------+------------+---------+
|Manhattan    |2015       |1           |green       |429801   |
|N/A          |2015       |1           |green       |844      |
|Staten Island|2015       |1           |green       |222      |
|EWR          |2015       |1           |green       |33       |
|Brooklyn     |2015       |1           |green       |569352   |
|Unknown      |2015       |1           |green       |2456     |
|Queens       |2015       |1           |green       |412150   |
|Bronx        |2015       |1           |green       |91410    |
|Manhattan    |2015       |1           |yellow      |11610665 |
|Queens       |2015       |1           |yellow      |636494   |
|N/A          |2015       |1           |yellow      |8153     |
|EWR          |2015       |1           |yellow      |687      |
|Staten Island|2015       |1           |

## k) Top 20 flujos PU→DO por volumen y ticket promedio

In [ ]:
result = (
    df.groupby("pu_zone", "do_zone")
      .agg(
          F.count("*").alias("num_trips"),
          F.round(F.avg("total_amount"), 2).alias("avg_total")
      )
      .orderBy(F.col("num_trips").desc())
      .limit(20)
)

result.show(truncate=False)

In [19]:
raw = query_obt("""
    SELECT pu_zone, do_zone,
           COUNT(*) as num_trips,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM OBT_TRIPS
    GROUP BY pu_zone, do_zone
""")

(raw.orderBy(F.col("num_trips").desc())
    .limit(20)
    .show(truncate=False)
)

+----------------------------+----------------------------+---------+---------+
|PU_ZONE                     |DO_ZONE                     |NUM_TRIPS|AVG_TOTAL|
+----------------------------+----------------------------+---------+---------+
|N/A                         |N/A                         |7674472  |17.82    |
|Upper East Side South       |Upper East Side North       |4583292  |10.5     |
|Upper East Side North       |Upper East Side South       |3918921  |11.4     |
|Upper East Side North       |Upper East Side North       |3610992  |8.78     |
|Upper East Side South       |Upper East Side South       |3462662  |9.39     |
|Upper West Side South       |Upper West Side North       |2034251  |9.02     |
|Upper West Side South       |Lincoln Square East         |2019507  |9.56     |
|Upper East Side South       |Midtown Center              |1963403  |12.33    |
|Upper East Side South       |Midtown East                |1950935  |10.89    |
|Lincoln Square East         |Upper West

In [20]:
query_obt("""
    SELECT pu_zone, do_zone,
           COUNT(*) as num_trips,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM OBT_TRIPS
    GROUP BY pu_zone, do_zone
    ORDER BY num_trips DESC
    LIMIT 20
""").show(truncate=False)

+----------------------------+----------------------------+---------+---------+
|PU_ZONE                     |DO_ZONE                     |NUM_TRIPS|AVG_TOTAL|
+----------------------------+----------------------------+---------+---------+
|N/A                         |N/A                         |7674472  |17.82    |
|Upper East Side South       |Upper East Side North       |4583292  |10.5     |
|Upper East Side North       |Upper East Side South       |3918921  |11.4     |
|Upper East Side North       |Upper East Side North       |3610992  |8.78     |
|Upper East Side South       |Upper East Side South       |3462662  |9.39     |
|Upper West Side South       |Upper West Side North       |2034251  |9.02     |
|Upper West Side South       |Lincoln Square East         |2019507  |9.56     |
|Upper East Side South       |Midtown Center              |1963403  |12.33    |
|Upper East Side South       |Midtown East                |1950935  |10.89    |
|Lincoln Square East         |Upper West

## l) Distribucion de passenger_count y efecto en total_amount

In [ ]:
w = Window.partitionBy() 

(
    df.groupby("passenger_count")
      .agg(
          F.count("*").alias("num_trips"),
          F.round(F.avg("total_amount"), 2).alias("avg_total")
      )
      .withColumn("pct",
          F.round(F.col("num_trips") * 100 / F.sum("num_trips").over(w), 2)
      )
      .orderBy("passenger_count")
      .show(truncate=False)
)

In [22]:
raw = query_obt("""
    SELECT passenger_count,
           COUNT(*) as num_trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM OBT_TRIPS
    GROUP BY passenger_count
""")

(raw.orderBy("passenger_count")
    .show(truncate=False)
)

+---------------+---------+-----+---------+
|PASSENGER_COUNT|NUM_TRIPS|PCT  |AVG_TOTAL|
+---------------+---------+-----+---------+
|NULL           |22828279 |2.63 |26.19    |
|0              |5944407  |0.69 |19.85    |
|1              |616191507|71.10|18.38    |
|2              |118784057|13.71|19.83    |
|3              |32857409 |3.79 |19.26    |
|4              |15951907 |1.84 |20.39    |
|5              |33512161 |3.87 |17.08    |
|6              |20595392 |2.38 |16.91    |
|7              |3845     |0.00 |47.66    |
|8              |3966     |0.00 |50.34    |
|9              |2048     |0.00 |62.77    |
|32             |1        |0.00 |60.35    |
|48             |1        |0.00 |40.3     |
|96             |2        |0.00 |12.59    |
|112            |1        |0.00 |14.76    |
|192            |2        |0.00 |9.15     |
+---------------+---------+-----+---------+



In [23]:
query_obt("""
    SELECT passenger_count,
           COUNT(*) as num_trips,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM OBT_TRIPS
    GROUP BY passenger_count
    ORDER BY passenger_count
""").show(truncate=False)

+---------------+---------+-----+---------+
|PASSENGER_COUNT|NUM_TRIPS|PCT  |AVG_TOTAL|
+---------------+---------+-----+---------+
|0              |5944407  |0.69 |19.85    |
|1              |616191507|71.10|18.38    |
|2              |118784057|13.71|19.83    |
|3              |32857409 |3.79 |19.26    |
|4              |15951907 |1.84 |20.39    |
|5              |33512161 |3.87 |17.08    |
|6              |20595392 |2.38 |16.91    |
|7              |3845     |0.00 |47.66    |
|8              |3966     |0.00 |50.34    |
|9              |2048     |0.00 |62.77    |
|32             |1        |0.00 |60.35    |
|48             |1        |0.00 |40.3     |
|96             |2        |0.00 |12.59    |
|112            |1        |0.00 |14.76    |
|192            |2        |0.00 |9.15     |
|NULL           |22828279 |2.63 |26.19    |
+---------------+---------+-----+---------+



## m) Impacto de tolls_amount y congestion_surcharge por zona

In [ ]:
result = (
    df.groupby("pu_zone")
      .agg(
          F.count("*").alias("num_trips"),
          F.round(F.avg("congestion_surcharge"), 2).alias("avg_congestion")
          F.round(F.avg("tolls_amount"), 2).alias("avg_tolls")
      )
      .orderBy(F.col("avg_congestion").desc())
)

result.show(truncate=False)

In [24]:
raw = query_obt("""
    SELECT pu_zone,
           ROUND(AVG(tolls_amount), 2) as avg_tolls,
           ROUND(AVG(congestion_surcharge), 2) as avg_congestion,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_zone
""")

(raw.orderBy(F.col("avg_congestion").desc())
    .show(truncate=False)
)

+-----------------------------+---------+--------------+---------+
|PU_ZONE                      |AVG_TOLLS|AVG_CONGESTION|NUM_TRIPS|
+-----------------------------+---------+--------------+---------+
|Greenwich Village South      |0.06     |2.96          |10948494 |
|Upper East Side South        |0.07     |2.96          |32918509 |
|West Village                 |0.1      |2.96          |16799124 |
|Little Italy/NoLiTa          |0.05     |2.95          |8580427  |
|Yorkville East               |0.2      |2.95          |9861745  |
|West Chelsea/Hudson Yards    |0.16     |2.95          |12151314 |
|Battery Park                 |0.2      |2.95          |331305   |
|SoHo                         |0.08     |2.95          |6581414  |
|Lincoln Square East          |0.11     |2.95          |23534454 |
|Union Sq                     |0.14     |2.95          |24407074 |
|East Village                 |0.09     |2.95          |22646202 |
|Sutton Place/Turtle Bay North|0.12     |2.95          |151486

In [25]:
query_obt("""
    SELECT pu_zone,
           ROUND(AVG(tolls_amount), 2) as avg_tolls,
           ROUND(AVG(congestion_surcharge), 2) as avg_congestion,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_zone
    ORDER BY avg_congestion DESC
""").show(truncate=False)

+-----------------------------+---------+--------------+---------+
|PU_ZONE                      |AVG_TOLLS|AVG_CONGESTION|NUM_TRIPS|
+-----------------------------+---------+--------------+---------+
|Upper East Side South        |0.07     |2.96          |32918509 |
|West Village                 |0.1      |2.96          |16799124 |
|Greenwich Village South      |0.06     |2.96          |10948494 |
|Little Italy/NoLiTa          |0.05     |2.95          |8580427  |
|Penn Station/Madison Sq West |0.19     |2.95          |26954732 |
|Battery Park                 |0.2      |2.95          |331305   |
|East Village                 |0.09     |2.95          |22646202 |
|Union Sq                     |0.14     |2.95          |24407074 |
|Greenwich Village North      |0.1      |2.95          |12628689 |
|Upper West Side South        |0.14     |2.95          |20528000 |
|Yorkville East               |0.2      |2.95          |9861745  |
|Lenox Hill West              |0.09     |2.95          |188019

## n) Proporcion viajes cortos vs largos por borough y estacionalidad

In [ ]:
result = (
    df.withColumn("season",
           F.when(F.col("month").isin(12, 1, 2), "Winter")
            .when(F.col("month").isin(3, 4, 5), "Spring")
            .when(F.col("month").isin(6, 7, 8), "Summer")
            .otherwise("Fall")
      )
      .groupby("pu_borough", "season")
      .agg(
          F.round(F.count("*"), 2).alias("total_trips"),
          F.round((F.sum(F.when(F.col("trip_distance") <= 2, 1).otherwise(0)) * 100) / F.count("*"), 2).alias("pct_short"),
          F.round((F.sum(F.when(F.col("trip_distance") > 10, 1).otherwise(0)) * 100) /  F.count("*"), 2).alias("pct_long")
      )
      .orderBy("pu_borough", "season")
)

result.show(truncate=False)

In [27]:
raw = query_obt("""
    SELECT pu_borough,
           CASE
               WHEN month IN (12, 1, 2) THEN 'Winter'
               WHEN month IN (3, 4, 5) THEN 'Spring'
               WHEN month IN (6, 7, 8) THEN 'Summer'
               ELSE 'Fall'
           END as season,
           ROUND(SUM(CASE WHEN trip_distance <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_short,
           ROUND(SUM(CASE WHEN trip_distance > 10 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_long,
           COUNT(*) as total_trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, season
""")

(raw.orderBy("pu_borough", "season")
    .show(truncate=False)
)

+----------+------+---------+--------+-----------+
|PU_BOROUGH|SEASON|PCT_SHORT|PCT_LONG|TOTAL_TRIPS|
+----------+------+---------+--------+-----------+
|Bronx     |Fall  |38.50    |13.65   |1201677    |
|Bronx     |Spring|42.16    |10.40   |1542694    |
|Bronx     |Summer|40.64    |12.00   |1285250    |
|Bronx     |Winter|41.72    |11.42   |1365025    |
|Brooklyn  |Fall  |41.46    |6.40    |8272466    |
|Brooklyn  |Spring|42.45    |5.50    |10005100   |
|Brooklyn  |Summer|40.90    |6.06    |8578802    |
|Brooklyn  |Winter|44.44    |5.57    |9039788    |
|EWR       |Fall  |89.73    |7.39    |20687      |
|EWR       |Spring|88.46    |8.04    |18624      |
|EWR       |Summer|89.25    |7.86    |18739      |
|EWR       |Winter|89.11    |7.74    |17770      |
|Manhattan |Fall  |61.70    |2.68    |182458696  |
|Manhattan |Spring|61.64    |2.63    |196031455  |
|Manhattan |Summer|60.91    |2.71    |172844469  |
|Manhattan |Winter|63.30    |2.40    |190223086  |
|N/A       |Fall  |53.55    |20

In [26]:
query_obt("""
    SELECT pu_borough,
           CASE
               WHEN month IN (12, 1, 2) THEN 'Winter'
               WHEN month IN (3, 4, 5) THEN 'Spring'
               WHEN month IN (6, 7, 8) THEN 'Summer'
               ELSE 'Fall'
           END as season,
           ROUND(SUM(CASE WHEN trip_distance <= 2 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_short,
           ROUND(SUM(CASE WHEN trip_distance > 10 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_long,
           COUNT(*) as total_trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, season
    ORDER BY pu_borough, season
""").show(truncate=False)

+----------+------+---------+--------+-----------+
|PU_BOROUGH|SEASON|PCT_SHORT|PCT_LONG|TOTAL_TRIPS|
+----------+------+---------+--------+-----------+
|Bronx     |Fall  |38.50    |13.65   |1201677    |
|Bronx     |Spring|42.16    |10.40   |1542694    |
|Bronx     |Summer|40.64    |12.00   |1285250    |
|Bronx     |Winter|41.72    |11.42   |1365025    |
|Brooklyn  |Fall  |41.46    |6.40    |8272466    |
|Brooklyn  |Spring|42.45    |5.50    |10005100   |
|Brooklyn  |Summer|40.90    |6.06    |8578802    |
|Brooklyn  |Winter|44.44    |5.57    |9039788    |
|EWR       |Fall  |89.73    |7.39    |20687      |
|EWR       |Spring|88.46    |8.04    |18624      |
|EWR       |Summer|89.25    |7.86    |18739      |
|EWR       |Winter|89.11    |7.74    |17770      |
|Manhattan |Fall  |61.70    |2.68    |182458696  |
|Manhattan |Spring|61.64    |2.63    |196031455  |
|Manhattan |Summer|60.91    |2.71    |172844469  |
|Manhattan |Winter|63.30    |2.40    |190223086  |
|N/A       |Fall  |53.55    |20

## o) Diferencias por vendor en avg_speed_mph y trip_duration_min

In [ ]:
result = (
    df.groupby("vendor_name")
      .agg(
          F.round(F.count("*"), 2).alias("num_trips"),
          F.round(F.avg("avg_speed_mph"), 2).alias("avg_speed"),
          F.round(F.avg("trip_duration_min"), 2).alias("avg_duration")
      )
      .orderBy(F.col("avg_speed").desc())
)

result.show(truncate=False)

In [28]:
raw = query_obt("""
    SELECT vendor_name,
           ROUND(AVG(avg_speed_mph), 2) as avg_speed,
           ROUND(AVG(trip_duration_min), 2) as avg_duration,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY vendor_name
""")

(raw.orderBy(F.col("avg_speed").desc())
    .show(truncate=False)
)

+---------------------------------+---------+------------+---------+
|VENDOR_NAME                      |AVG_SPEED|AVG_DURATION|NUM_TRIPS|
+---------------------------------+---------+------------+---------+
|Myle Technologies Inc            |702.66   |45.4        |216509   |
|Creative Mobile Technologies, LLC|69.64    |16.95       |325832862|
|Curb Mobility, LLC               |17.85    |21.09       |539319223|
|Unknown                          |10.73    |14.18       |770260   |
|Helix                            |NULL     |0.0         |536131   |
+---------------------------------+---------+------------+---------+



In [29]:
query_obt("""
    SELECT vendor_name,
           ROUND(AVG(avg_speed_mph), 2) as avg_speed,
           ROUND(AVG(trip_duration_min), 2) as avg_duration,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY vendor_name
    ORDER BY avg_speed DESC
""").show(truncate=False)

+---------------------------------+---------+------------+---------+
|VENDOR_NAME                      |AVG_SPEED|AVG_DURATION|NUM_TRIPS|
+---------------------------------+---------+------------+---------+
|Helix                            |NULL     |0.0         |536131   |
|Myle Technologies Inc            |702.66   |45.4        |216509   |
|Creative Mobile Technologies, LLC|69.64    |16.95       |325832862|
|Curb Mobility, LLC               |17.85    |21.09       |539319223|
|Unknown                          |10.73    |14.18       |770260   |
+---------------------------------+---------+------------+---------+



## p) Relacion metodo de pago y tip_amount por hora

In [ ]:
result = (
    df.groupby("payment_type_desc", "pickup_hour")
      .agg(
          F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
          F.round(F.count("*"), 2).alias("num_trips")
      )
      .orderBy("payment_type_desc", "pickup_hour")
)

result.show(truncate=False)

In [35]:
raw = query_obt("""
    SELECT payment_type_desc, pickup_hour,
           ROUND(AVG(tip_amount), 2) as avg_tip,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY payment_type_desc, pickup_hour
""")

(raw.orderBy(F.col("avg_tip").desc())
    .show(truncate=False)
)

+-----------------+-----------+-------+---------+
|PAYMENT_TYPE_DESC|PICKUP_HOUR|AVG_TIP|NUM_TRIPS|
+-----------------+-----------+-------+---------+
|Flex Fare trip   |9          |54.43  |834700   |
|Unknown          |12         |6.3    |159      |
|Credit card      |5          |3.81   |4738599  |
|Credit card      |16         |3.44   |29281527 |
|Credit card      |4          |3.34   |4388366  |
|Credit card      |15         |3.26   |30558349 |
|Credit card      |14         |3.25   |30480606 |
|Credit card      |17         |3.23   |33889241 |
|Credit card      |13         |3.15   |28854489 |
|Credit card      |23         |3.14   |26066593 |
|Credit card      |22         |3.11   |31842540 |
|Credit card      |21         |3.07   |33910148 |
|Credit card      |19         |3.06   |37293446 |
|Credit card      |12         |3.05   |28681661 |
|Credit card      |0          |3.04   |19130590 |
|Credit card      |18         |3.03   |38500870 |
|Credit card      |11         |3.0    |27077948 |


In [34]:
query_obt("""
    SELECT payment_type_desc, pickup_hour,
           ROUND(AVG(tip_amount), 2) as avg_tip,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY payment_type_desc, pickup_hour
    ORDER BY payment_type_desc, pickup_hour
""").show(truncate=False)

+-----------------+-----------+-------+---------+
|PAYMENT_TYPE_DESC|PICKUP_HOUR|AVG_TIP|NUM_TRIPS|
+-----------------+-----------+-------+---------+
|Cash             |0          |0.0    |8256208  |
|Cash             |1          |0.0    |6060179  |
|Cash             |2          |0.0    |4469512  |
|Cash             |3          |0.0    |3488262  |
|Cash             |4          |0.0    |3193817  |
|Cash             |5          |0.0    |2874725  |
|Cash             |6          |0.0    |5304742  |
|Cash             |7          |0.0    |7895408  |
|Cash             |8          |0.0    |9474311  |
|Cash             |9          |0.0    |10923693 |
|Cash             |10         |0.0    |12476760 |
|Cash             |11         |0.0    |13525361 |
|Cash             |12         |0.0    |14346719 |
|Cash             |13         |0.0    |14662985 |
|Cash             |14         |0.0    |15484012 |
|Cash             |15         |0.0    |15383773 |
|Cash             |16         |0.0    |14163665 |


## q) Zonas con percentil 99 de duracion/distancia fuera de rango

In [ ]:
result = (
    df.groupby("pu_zone")
      .agg(
          F.round(F.percentile_approx("trip_duration_min", 0.99), 2).alias("p99_duration"),
          F.round(F.percentile_approx("trip_distance", 0.99), 2).alias("p99_distance"),
          F.round(F.count("*"), 2).alias("num_trips")
      ) 
      .orderBy(F.col("p99_duration").desc())
)

result.show(truncate=False)

In [36]:
raw = query_obt("""
    SELECT pu_zone,
           ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p99_duration,
           ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance), 2) AS p99_distance,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_zone
""")

(raw.orderBy(F.col("p99_duration").desc())
    .show(truncate=False)
)

+---------------------------------+------------+------------+---------+
|PU_ZONE                          |P99_DURATION|P99_DISTANCE|NUM_TRIPS|
+---------------------------------+------------+------------+---------+
|Arden Heights                    |184.38      |40.13       |2227     |
|Saint Michaels Cemetery/Woodside |176.0       |13.81       |61519    |
|Bronx Park                       |175.43      |21.96       |34050    |
|Coney Island                     |168.92      |29.74       |197866   |
|Mariners Harbor                  |154.9       |36.13       |5146     |
|Rossville/Woodrow                |152.12      |45.79       |399      |
|Eltingville/Annadale/Prince's Bay|144.74      |47.38       |554      |
|Hammels/Arverne                  |140.19      |31.85       |43476    |
|Far Rockaway                     |139.79      |30.9        |38427    |
|Rockaway Park                    |130.1       |31.12       |11780    |
|Heartland Village/Todt Hill      |128.41      |39.19       |268

In [37]:
query_obt("""
    SELECT pu_zone,
           ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_duration_min), 2) AS p99_duration,
           ROUND(PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance), 2) AS p99_distance,
           COUNT(*) as num_trips
    FROM OBT_TRIPS
    GROUP BY pu_zone
    ORDER BY p99_duration DESC
""").show(truncate=False)

+---------------------------------+------------+------------+---------+
|PU_ZONE                          |P99_DURATION|P99_DISTANCE|NUM_TRIPS|
+---------------------------------+------------+------------+---------+
|Arden Heights                    |184.38      |40.13       |2227     |
|Saint Michaels Cemetery/Woodside |176.0       |13.81       |61519    |
|Bronx Park                       |175.43      |21.96       |34050    |
|Coney Island                     |168.92      |29.74       |197866   |
|Mariners Harbor                  |154.9       |36.13       |5146     |
|Rossville/Woodrow                |152.12      |45.79       |399      |
|Eltingville/Annadale/Prince's Bay|144.74      |47.38       |554      |
|Hammels/Arverne                  |140.19      |31.85       |43476    |
|Far Rockaway                     |139.79      |30.9        |38427    |
|Rockaway Park                    |130.1       |31.12       |11780    |
|Heartland Village/Todt Hill      |128.41      |39.19       |268

## r) Yield por milla (total_amount/trip_distance) por borough y hora

In [ ]:
result = (
    df.withColumn("yield_per_mile",
           F.when(F.col("trip_distance") > 0,
                  F.col("total_amount") / F.col("trip_distance"))
                 )
      .groupby("pu_borough", "pickup_hour")
      .agg(
          F.round(F.count("*"), 2).alias("num_trips"),
          F.round(F.avg("yield_per_mile"), 2).alias("avg_yield_per_mile")
      ) 
      .orderBy("pu_borough", "pickup_hour")
)

result.show(truncate=False)

In [38]:
raw = query_obt("""
    SELECT pu_borough, pickup_hour,
           ROUND(AVG(total_amount / NULLIF(trip_distance, 0)), 2) as yield_per_mile,
           COUNT(*) as trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, pickup_hour
""")

(raw.orderBy("pu_borough", "pickup_hour")
    .show(truncate=False)
)

+----------+-----------+--------------+------+
|PU_BOROUGH|PICKUP_HOUR|YIELD_PER_MILE|TRIPS |
+----------+-----------+--------------+------+
|Bronx     |0          |14.17         |145419|
|Bronx     |1          |16.62         |108176|
|Bronx     |2          |17.86         |77826 |
|Bronx     |3          |15.72         |67858 |
|Bronx     |4          |13.7          |86111 |
|Bronx     |5          |12.22         |107142|
|Bronx     |6          |11.37         |178542|
|Bronx     |7          |11.06         |285863|
|Bronx     |8          |11.84         |348256|
|Bronx     |9          |11.94         |323016|
|Bronx     |10         |11.89         |290300|
|Bronx     |11         |11.7          |276234|
|Bronx     |12         |11.18         |280472|
|Bronx     |13         |12.19         |280711|
|Bronx     |14         |12.16         |302498|
|Bronx     |15         |12.77         |309948|
|Bronx     |16         |17.93         |305035|
|Bronx     |17         |19.56         |310980|
|Bronx     |1

In [40]:
query_obt("""
    SELECT pu_borough, pickup_hour,
           ROUND(AVG(total_amount / NULLIF(trip_distance, 0)), 2) as yield_per_mile,
           COUNT(*) as trips
    FROM OBT_TRIPS
    GROUP BY pu_borough, pickup_hour
    ORDER BY pu_borough, pickup_hour
""").show(truncate=False)

+----------+-----------+--------------+------+
|PU_BOROUGH|PICKUP_HOUR|YIELD_PER_MILE|TRIPS |
+----------+-----------+--------------+------+
|Bronx     |0          |14.17         |145419|
|Bronx     |1          |16.62         |108176|
|Bronx     |2          |17.86         |77826 |
|Bronx     |3          |15.72         |67858 |
|Bronx     |4          |13.7          |86111 |
|Bronx     |5          |12.22         |107142|
|Bronx     |6          |11.37         |178542|
|Bronx     |7          |11.06         |285863|
|Bronx     |8          |11.84         |348256|
|Bronx     |9          |11.94         |323016|
|Bronx     |10         |11.89         |290300|
|Bronx     |11         |11.7          |276234|
|Bronx     |12         |11.18         |280472|
|Bronx     |13         |12.19         |280711|
|Bronx     |14         |12.16         |302498|
|Bronx     |15         |12.77         |309948|
|Bronx     |16         |17.93         |305035|
|Bronx     |17         |19.56         |310980|
|Bronx     |1

## s) Cambios YoY en volumen y ticket promedio por service_type

In [39]:
yearly = (
    df.groupby("service_type", "source_year")
      .agg(
          F.count("*").alias("trips"),
          F.round(F.avg("total_amount"), 2).alias("avg_total")
      )
)

a = yearly.alias("a")
b = yearly.alias("b")

result = (
    a.join(
        b,
        (F.col("a.service_type") == F.col("b.service_type")) &
        (F.col("a.source_year") == F.col("b.source_year") + 1),
        "left"
    )
    .select(
        F.col("a.service_type"),
        F.col("a.source_year"),
        F.col("a.trips"),
        F.col("a.avg_total"),
        F.round(
            (F.col("a.trips") - F.col("b.trips")) * 100 /
            F.when(F.col("b.trips") != 0, F.col("b.trips")), 2
        ).alias("volume_change_pct"),
        F.round(
            F.col("a.avg_total") - F.col("b.avg_total"), 2
        ).alias("ticket_change")
    )
    .orderBy("service_type", "source_year")
)

result.show(truncate=False)



In [43]:
raw = query_obt("""
    WITH yearly AS (
        SELECT service_type, source_year,
               COUNT(*) as trips,
               ROUND(AVG(total_amount), 2) as avg_total
        FROM OBT_TRIPS
        GROUP BY service_type, source_year
    )
    SELECT
        a.service_type, a.source_year,
        a.trips, a.avg_total,
        ROUND((a.trips - b.trips) * 100.0 / NULLIF(b.trips, 0), 2) as volume_change_pct,
        ROUND(a.avg_total - b.avg_total, 2) as ticket_change
    FROM yearly a
    LEFT JOIN yearly b ON a.service_type = b.service_type AND a.source_year = b.source_year + 1
    ORDER BY a.service_type, a.source_year
""")

(raw.show(truncate=False)
)

+------------+-----------+---------+---------+-----------------+-------------+
|SERVICE_TYPE|SOURCE_YEAR|TRIPS    |AVG_TOTAL|VOLUME_CHANGE_PCT|TICKET_CHANGE|
+------------+-----------+---------+---------+-----------------+-------------+
|green       |2015       |19202652 |14.88    |NULL             |NULL         |
|green       |2016       |16352455 |14.69    |-14.84           |-0.19        |
|green       |2017       |11710371 |14.29    |-28.39           |-0.4         |
|green       |2018       |8876633  |16.15    |-24.20           |1.86         |
|green       |2019       |6262384  |18.37    |-29.45           |2.22         |
|green       |2020       |1728859  |20.23    |-72.39           |1.86         |
|green       |2021       |1066639  |24.0     |-38.30           |3.77         |
|green       |2022       |838206   |19.4     |-21.42           |-4.6         |
|green       |2023       |784821   |23.96    |-6.37            |4.56         |
|green       |2024       |658042   |24.39    |-16.15

In [42]:
query_obt("""
    WITH yearly AS (
        SELECT service_type, source_year,
               COUNT(*) as trips,
               ROUND(AVG(total_amount), 2) as avg_total
        FROM OBT_TRIPS
        GROUP BY service_type, source_year
    )
    SELECT
        a.service_type, a.source_year,
        a.trips, a.avg_total,
        ROUND((a.trips - b.trips) * 100.0 / NULLIF(b.trips, 0), 2) as volume_change_pct,
        ROUND(a.avg_total - b.avg_total, 2) as ticket_change
    FROM yearly a
    LEFT JOIN yearly b ON a.service_type = b.service_type AND a.source_year = b.source_year + 1
    ORDER BY a.service_type, a.source_year
""").show(truncate=False)

+------------+-----------+---------+---------+-----------------+-------------+
|SERVICE_TYPE|SOURCE_YEAR|TRIPS    |AVG_TOTAL|VOLUME_CHANGE_PCT|TICKET_CHANGE|
+------------+-----------+---------+---------+-----------------+-------------+
|green       |2015       |19202652 |14.88    |NULL             |NULL         |
|green       |2016       |16352455 |14.69    |-14.84           |-0.19        |
|green       |2017       |11710371 |14.29    |-28.39           |-0.4         |
|green       |2018       |8876633  |16.15    |-24.20           |1.86         |
|green       |2019       |6262384  |18.37    |-29.45           |2.22         |
|green       |2020       |1728859  |20.23    |-72.39           |1.86         |
|green       |2021       |1066639  |24.0     |-38.30           |3.77         |
|green       |2022       |838206   |19.4     |-21.42           |-4.6         |
|green       |2023       |784821   |23.96    |-6.37            |4.56         |
|green       |2024       |658042   |24.39    |-16.15

## t) Dias con alta congestion_surcharge: efecto en total_amount vs dias normales

In [ ]:
daily = (
    df.filter(
        (F.col("congestion_surcharge").isNotNull()) &
        (F.col("pickup_date").isNotNull())
    )
    .groupby("pickup_date")
    .agg(
        F.avg("congestion_surcharge").alias("avg_congestion"),
        F.avg("total_amount").alias("avg_total"),
        F.count("*").alias("num_trips")
    )
)

# 2. Clasificación + agregación final
result = (
    daily.withColumn(
        "congestion_level",
        F.when(F.col("avg_congestion") > 2.0, "Alta congestion")
         .otherwise("Normal")
    )
    .groupby("congestion_level")
    .agg(
        F.count("*").alias("num_days"),
        F.round(F.avg("avg_total"), 2).alias("avg_total_amount"),
        F.round(F.avg("num_trips"), 0).alias("avg_daily_trips")
    )
)

result.show(truncate=False)

In [ ]:
raw = query_obt("""
    WITH daily AS (
        SELECT pickup_date,
               AVG(congestion_surcharge) as avg_congestion,
               AVG(total_amount) as avg_total,
               COUNT(*) as num_trips
        FROM OBT_TRIPS
        WHERE congestion_surcharge IS NOT NULL AND pickup_date IS NOT NULL
        GROUP BY pickup_date
    )
    SELECT
        CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END as congestion_level,
        COUNT(*) as num_days,
        ROUND(AVG(avg_total), 2) as avg_total_amount,
        ROUND(AVG(num_trips)) as avg_daily_trips
    FROM daily
    GROUP BY CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END
""")

(raw.show(truncate=False)
)

In [47]:
query_obt("""
    WITH daily AS (
        SELECT pickup_date,
               AVG(congestion_surcharge) as avg_congestion,
               AVG(total_amount) as avg_total,
               COUNT(*) as num_trips
        FROM OBT_TRIPS
        WHERE congestion_surcharge IS NOT NULL AND pickup_date IS NOT NULL
        GROUP BY pickup_date
    )
    SELECT
        CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END as congestion_level,
        COUNT(*) as num_days,
        ROUND(AVG(avg_total), 2) as avg_total_amount,
        ROUND(AVG(num_trips)) as avg_daily_trips
    FROM daily
    GROUP BY CASE WHEN avg_congestion > 2.0 THEN 'Alta congestion' ELSE 'Normal' END
""").show(truncate=False)

+----------------+--------+----------------+---------------+
|CONGESTION_LEVEL|NUM_DAYS|AVG_TOTAL_AMOUNT|AVG_DAILY_TRIPS|
+----------------+--------+----------------+---------------+
|Alta congestion |2550    |23.62           |112134         |
|Normal          |53      |12.7            |60837          |
+----------------+--------+----------------+---------------+

